In [ ]:
# Comparison of Antibiotic Resistance Genes (ARGs) Across Five Wastewater Samples (R version)
# Step 1: Install and Load Required Packages
# Install once (takes about a minute)
if (!requireNamespace("pheatmap", quietly = TRUE)) install.packages("pheatmap")

library(pheatmap)

In [ ]:
# Step 2: Read and Extract Unique AMR Genes from Files
files <- sort(list.files(pattern = "unique.*\\.csv$", ignore.case = TRUE))
print(files)

file_dict <- list()

for (f in files) {
  # Sample name from file name: "A(1) unique .csv" -> "A1", "K unique .csv" -> "K"
  sample_name <- sub("[ _]*unique.*$", "", f, ignore.case = TRUE)
  sample_name <- gsub("[^A-Za-z0-9]", "", sample_name)

  lines <- readLines(f, warn = FALSE, encoding = "UTF-8")

  # Find the header line and detect the delimiter (; or ,)
  header_idx <- which(grepl("Gene", lines))[1]
  if (is.na(header_idx)) { warning(paste("No header found in", f)); next }
  delim <- if (grepl(";", lines[header_idx], fixed = TRUE)) ";" else ","

  df <- read.table(text = lines[header_idx:length(lines)],
                   sep = delim, header = TRUE, quote = "\"",
                   comment.char = "", fill = TRUE, check.names = FALSE,
                   stringsAsFactors = FALSE, strip.white = TRUE)
  names(df) <- trimws(names(df))

  total_rows <- sum(trimws(df$Type) != "")          # ignore empty rows
  df <- df[trimws(df$Type) == "AMR", ]              # keep AMR only (remove STRESS/BIOCIDE)

  genes <- unique(trimws(df$Gene))                  # unique within the sample, for the matrix only
  genes <- genes[!is.na(genes) & genes != "" & tolower(genes) != "gene"]

  file_dict[[sample_name]] <- genes
  cat(sprintf("%-3s | file: %-22s | records: %3d | AMR records: %3d | unique AMR genes: %3d\n",
              sample_name, f, total_rows, nrow(df), length(genes)))
}

# Fixed column order
sample_order <- intersect(c("A1", "A2", "B1", "B2", "K"), names(file_dict))
file_dict <- file_dict[sample_order]

In [ ]:
# Step 3: Create Master Gene List Across All Samples
# Each unique gene appears once; original gene names are kept unchanged
all_genes <- sort(unique(unlist(file_dict)), method = "radix")
cat("Total unique AMR genes:", length(all_genes), "\n")

In [ ]:
# Step 4: Build Presence/Absence Matrix (0 and 1 Values)
mat <- sapply(file_dict, function(genes) as.integer(all_genes %in% genes))
rownames(mat) <- all_genes

df_matrix <- data.frame(Gene = all_genes, mat, check.names = FALSE, row.names = NULL)
head(df_matrix, 10)

In [ ]:
# Step 5: Save Matrix as CSV and Display Table
write.csv(df_matrix, "arg_presence_absence.csv", row.names = FALSE)   # Gene in the first column
print(dim(df_matrix))
df_matrix

In [ ]:
# Step 6: Check the Matrix Against the Original Sample Results
# Verification: number of unique AMR genes per sample must match the matrix column sums
check <- data.frame(Sample = names(file_dict),
                    Unique_AMR_genes = sapply(file_dict, length),
                    Matrix_column_sum = colSums(mat),
                    row.names = NULL)
check$Match <- check$Unique_AMR_genes == check$Matrix_column_sum
print(check)

stopifnot(all(check$Match))
cat("\nAll samples match.\n")

# Every gene in the matrix must be present in at least one sample
stopifnot(all(rowSums(mat) >= 1))

In [ ]:
# Step 7: Heatmap of ARG Presence/Absence (pheatmap)
m <- as.matrix(read.csv("arg_presence_absence.csv", row.names = 1, check.names = FALSE))

plot_height <- max(8, 0.17 * nrow(m))

# Save high-resolution PNG
pheatmap(m,
         cluster_rows = FALSE, cluster_cols = FALSE,
         color = c("#4575b4", "#d73027"),            # blue = 0, red = 1
         legend_breaks = c(0, 1),
         legend_labels = c("Not detected (0)", "Detected (1)"),
         border_color = "white",
         fontsize_row = 6, fontsize_col = 10,
         main = "ARG Presence/Absence Heatmap Across the Five Wastewater Samples",
         filename = "arg_presence_absence_heatmap.png",
         width = 7, height = plot_height)

# Show it in the notebook
options(repr.plot.width = 7, repr.plot.height = plot_height)
pheatmap(m,
         cluster_rows = FALSE, cluster_cols = FALSE,
         color = c("#4575b4", "#d73027"),
         legend_breaks = c(0, 1),
         legend_labels = c("Not detected (0)", "Detected (1)"),
         border_color = "white",
         fontsize_row = 6, fontsize_col = 10,
         main = "ARG Presence/Absence Heatmap Across the Five Wastewater Samples")

# exploratory clustering of samples
pheatmap(m,
         cluster_rows = FALSE, cluster_cols = TRUE,
         clustering_distance_cols = "binary",
         color = c("#4575b4", "#d73027"),
         legend_breaks = c(0, 1),
         legend_labels = c("Not detected (0)", "Detected (1)"),
         border_color = "white", fontsize_row = 6,
         main = "ARG Presence/Absence (samples clustered)")